# 08 · Monitoring

**Role of this layer.** Every stage of every run (and every replay) appends one
JSON object to `runs/events.jsonl`. The log is append-only, so it doubles as an
audit trail and could be shipped to a log platform as is.

The Monitoring page reads two sources, and the functions below (the ones behind
the page) turn them into numbers:
- the **run records**, one per run in its final state: activity, human review,
  tokens and cost, latency and failures;
- the **event log**, for what can happen several times in a run: guardrail
  firings, capability executions and replays.

To make sure there is something to show, this notebook first runs the five example
workflows in mock mode (approving only 1 and 2, as `cli demo` does).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.mock_scenarios import SCENARIOS

for scenario in SCENARIOS:
    session = run_request(
        scenario.request, list(scenario.dataset_ids), "mock", approve=scenario.auto_approve
    )
    print(scenario.id, session.status)

volatility_flag_success completed


vix_full_sample_lookahead completed
arima_forecast_block blocked
prompt_injection_block blocked
spread_frequency_mismatch_block blocked


In [3]:
from quantlab.lineage import load_runs
from quantlab.monitoring import activity, read_events

records = load_runs(settings.runs_dir)
events = read_events(settings.events_path)
print(len(records), "run records,", len(events), "events")
pd.Series(activity(records))

14 run records, 182 events


total        14
completed     7
blocked       6
failed        0
rejected      0
awaiting      1
dtype: int64

Quality and governance. Approval, edit rate and approval time count only plans
reviewed by a person on the Analysis page: these headless runs are left out
(`headless`), so those metrics are empty here.

In [4]:
from quantlab.monitoring import (
    narrative_failures,
    planner_refusals,
    replay_results,
    review_outcomes,
)

pd.Series(
    {
        **review_outcomes(records),
        **replay_results(events),
        **narrative_failures(records),
        **planner_refusals(records),
    }
)

shown                 0.0000
approved              0.0000
rejected              0.0000
abandoned             0.0000
approval_rate            NaN
edit_rate                NaN
approval_median_s        NaN
approval_p95_s           NaN
headless              6.0000
replays               1.0000
replay_match_rate     1.0000
narratives            7.0000
n01_failure_rate      0.0000
planned              14.0000
p01_refusal_rate      0.2857
dtype: float64

Every guardrail and every capability, zeros included, most frequent first:

In [5]:
from quantlab.monitoring import capability_counts, guardrail_counts

pd.DataFrame(guardrail_counts(events))

,code,name,action,count
0,V10,MATERIAL_ASSUMPTION,APPROVE,28
1,P01,PLANNER_REFUSED,BLOCK,4
2,V09,LOOK_AHEAD,APPROVE,3
3,V03,INVALID_PARAM,BLOCK,2
4,V08,FREQUENCY_MISMATCH,BLOCK,2
5,R01,REQUEST_TOO_LONG,BLOCK,0
6,V01,INVALID_PLAN,BLOCK,0
7,V02,UNKNOWN_CAPABILITY,BLOCK,0
8,V04,TYPE_MISMATCH,BLOCK,0
9,V05,BUDGET_EXCEEDED,BLOCK,0


In [6]:
pd.DataFrame(capability_counts(events))

,capability,code,count
0,load_series,C01,7
1,flag_regime,C07,7
2,resample,C02,5
3,compute_returns,C03,5
4,rolling_volatility,C06,5
5,forward_return,C08,5
6,compare_groups,C09,5
7,hac_mean_difference,C10,5
8,difference,C04,0
9,spread,C05,0


Cost per run, from the tokens the API reported and the prices in
`config.LLM_PRICES_PER_MILLION_TOKENS`. Mock runs call no LLM and are left out, so
in mock mode the table has no runs and the cache saved nothing:

In [7]:
from quantlab.config import LLM_PRICES_PER_MILLION_TOKENS
from quantlab.monitoring import cache_savings, cost_table

display(pd.DataFrame(cost_table(records, LLM_PRICES_PER_MILLION_TOKENS)))
pd.Series(cache_savings(records, LLM_PRICES_PER_MILLION_TOKENS))

,stage,runs,input_mean,input_p95,output_mean,output_p95,cost_mean,cost_p95,cost_total
0,planner,0,None,None,None,None,None,None,None
1,narrator,0,None,None,None,None,None,None,None
2,total,0,None,None,None,None,None,None,None


planner_calls          1.0
cache_hits             0.0
hit_rate               0.0
saved_input_tokens     0.0
saved_output_tokens    0.0
saved_cost             0.0
dtype: float64

Latency per stage (mean and 95th percentile, milliseconds), and the most recent
failures with the stage and the guardrail that stopped them:

In [8]:
from quantlab.monitoring import latency_table, recent_failures

pd.DataFrame(latency_table(records))

,stage,runs,mean_ms,p95_ms
0,planning,14,559.771429,2742.8800
1,validation,14,0.114286,0.2005
2,execution,7,118.980000,151.2880
3,narration,7,441.357143,2162.6500


In [9]:
pd.DataFrame(recent_failures(records, limit=5))

,time,run_id,status,stage,code,message
0,2026-10-18T15:42:45.035Z,20261018-154245-c878dd,blocked,Planner,P01,The plan has blocking issues: P01:plan
1,2026-10-18T15:42:45.073Z,20261018-154245-7f7acf,blocked,Planner,P01,The plan has blocking issues: P01:plan
2,2026-10-18T15:42:45.107Z,20261018-154245-2a8d67,blocked,Validator,V08,The plan has blocking issues: V08:s4
3,2026-10-18T13:36:12.006Z,20261018-133612-e047cb,blocked,Planner,P01,The plan has blocking issues: P01:plan
4,2026-10-18T13:36:12.040Z,20261018-133612-c2fe25,blocked,Validator,V08,The plan has blocking issues: V08:s4
